# HydrAI: Tree-Based Model Training

This notebook trains **tree-based** ML surrogate models on PFR training data: 
Random Forest, Gradient Boosting, XGBoost, and AdaBoost.

## Steps
1. **Setup** – Path and imports (RF, GB, XGBoost, AdaBoost)  
2. **Load data** – From `data/training/` (pkl)  
3. **Prepare features and targets** – Same as `model_training.py`  
4. **Train tree models** – One model per target for each type (RF, GB, XGB, AdaBoost)  
5. **Evaluate** – R², MSE, MAE per model type; save to `models/`

In [ ]:
# Setup and imports (tree-based models only)
import os
import sys
import glob
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import xgboost as xgb

# Project root: notebooks live under notebooks/; from there, project root is one level up
current_dir = Path(os.getcwd())
project_root = current_dir if (current_dir / 'src').exists() else current_dir.parent
sys.path.insert(0, str(project_root))
os.chdir(project_root)

print(f"Project root: {project_root}")
print("Imports OK.")

In [ ]:
# Config and data path
CONFIG_PATH = project_root / "configs" / "ml_training_config.json"
DATA_DIR = project_root / "data" / "training"
MODELS_DIR = project_root / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Load config (tree-specific: random_forest, gradient_boosting, xgboost, adaboost)
if CONFIG_PATH.exists():
    with open(CONFIG_PATH) as f:
        config = json.load(f)
    TEST_SIZE = config.get("test_size", 0.2)
    RANDOM_STATE = config.get("random_state", 42)
    RF_CONFIG = config.get("random_forest", {})
    GB_CONFIG = config.get("gradient_boosting", {})
    XGB_CONFIG = config.get("xgboost", {})
    ADA_CONFIG = config.get("adaboost", {})
else:
    TEST_SIZE, RANDOM_STATE = 0.2, 42
    RF_CONFIG = {"n_estimators": 100, "max_depth": 20}
    GB_CONFIG = {"n_estimators": 100, "max_depth": 5}
    XGB_CONFIG = {"n_estimators": 100, "max_depth": 6}
    ADA_CONFIG = {"n_estimators": 50, "learning_rate": 1.0, "max_depth": 3}

# Find latest training data (pkl only)
pkl_files = sorted(glob.glob(str(DATA_DIR / "training_data_complete_*.pkl")), reverse=True)
if pkl_files:
    DATA_FILE = pkl_files[0]
    df = pd.read_pickle(DATA_FILE)
else:
    raise FileNotFoundError(f"No training data (.pkl) in {DATA_DIR}. Run Main_2_generate_training_data first.")

print(f"Data: {Path(DATA_FILE).name}")
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")

In [ ]:
# Features and targets (same as model_training.py) 
# DUMMY FOR NOW !!!
feature_cols = [
    "initial_temperature_K",
    "initial_pressure_Pa",
    "reactor_length_m",
    "reactor_diameter_m",
    "mass_flow_rate_kgps",
    "heat_flux_Wm2",
    "z_position_m",
    "relative_position",
]
if "reactant_type" in df.columns:
    feature_cols.append("reactant_type")

primary_targets = ["temperature_K", "pressure_Pa", "velocity_ms", "density_kgm3"]
feature_cols = [c for c in feature_cols if c in df.columns]
target_cols = [c for c in primary_targets if c in df.columns]

X = df[feature_cols].copy()
le = None
if "reactant_type" in X.columns:
    le = LabelEncoder()
    X["reactant_type"] = le.fit_transform(X["reactant_type"].astype(str))
y = df[target_cols].copy()

# Drop rows with NaN
valid = ~(X.isna().any(axis=1) | y.isna().any(axis=1))
X, y = X[valid], y[valid]
print(f"Features: {feature_cols}")
print(f"Targets: {target_cols}")
print(f"Valid samples: {len(X)}")

In [ ]:
# Train/test split and scale features
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
scaler_X = StandardScaler()
X_train_s = scaler_X.fit_transform(X_train)
X_test_s = scaler_X.transform(X_test)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

In [ ]:
# Train tree-based models: one model per target for each type (RF, GB, XGBoost, AdaBoost)
all_models = {}  # key: "random_forest" | "gradient_boosting" | "xgboost" | "adaboost" -> dict of {target: model}

# 1. Random Forest
rf_models = {}
for col in target_cols:
    m = RandomForestRegressor(
        n_estimators=RF_CONFIG.get("n_estimators", 100),
        max_depth=RF_CONFIG.get("max_depth", 20),
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    m.fit(X_train_s, y_train[col])
    rf_models[col] = m
all_models["random_forest"] = rf_models
print("Random Forest: done.")

# 2. Gradient Boosting
gb_models = {}
for col in target_cols:
    m = GradientBoostingRegressor(
        n_estimators=GB_CONFIG.get("n_estimators", 100),
        max_depth=GB_CONFIG.get("max_depth", 5),
        random_state=RANDOM_STATE,
    )
    m.fit(X_train_s, y_train[col])
    gb_models[col] = m
all_models["gradient_boosting"] = gb_models
print("Gradient Boosting: done.")

# 3. XGBoost
xgb_models = {}
for col in target_cols:
    m = xgb.XGBRegressor(
        n_estimators=XGB_CONFIG.get("n_estimators", 100),
        max_depth=XGB_CONFIG.get("max_depth", 6),
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    m.fit(X_train_s, y_train[col])
    xgb_models[col] = m
all_models["xgboost"] = xgb_models
print("XGBoost: done.")

# 4. AdaBoost (tree-based: base estimator = DecisionTreeRegressor)
ada_models = {}
for col in target_cols:
    base = DecisionTreeRegressor(
        max_depth=ADA_CONFIG.get("max_depth", 3),
        random_state=RANDOM_STATE,
    )
    m = AdaBoostRegressor(
        estimator=base,
        n_estimators=ADA_CONFIG.get("n_estimators", 50),
        learning_rate=ADA_CONFIG.get("learning_rate", 1.0),
        random_state=RANDOM_STATE,
    )
    m.fit(X_train_s, y_train[col])
    ada_models[col] = m
all_models["adaboost"] = ada_models
print("AdaBoost: done.")

print("Tree-based training complete.")

In [ ]:
# Evaluate each tree model type
print("Test set performance (tree models):")
print("=" * 60)
for model_name, models_dict in all_models.items():
    print(f"\n  {model_name.upper()}")
    print("  " + "-" * 50)
    for col in target_cols:
        pred = models_dict[col].predict(X_test_s)
        r2 = r2_score(y_test[col], pred)
        mse = mean_squared_error(y_test[col], pred)
        mae = mean_absolute_error(y_test[col], pred)
        print(f"    {col}: R²={r2:.4f}, MSE={mse:.4e}, MAE={mae:.4e}")
print("\n" + "=" * 60)

In [ ]:
# Save each tree model type (scaler and encoders shared)
for model_name, models_dict in all_models.items():
    artifact = {
        "models": models_dict,
        "scaler_X": scaler_X,
        "feature_cols": feature_cols,
        "target_cols": target_cols,
        "label_encoder": le,
        "model_type": model_name,
    }
    out_path = MODELS_DIR / f"{model_name}_primary.joblib"
    joblib.dump(artifact, out_path)
    print(f"Saved: {out_path}")
print("Tree model artifacts saved.")